In [1]:
import torch
from diffusion_networks import SongUNet
from dataset import SQGLeadTimeDataset, DATA_500
from torch.utils.data import DataLoader



model = SongUNet(
        img_resolution     = 64,
        in_channels        = 2 * 2,  # 2 (z_s) + 2 (x_t conditioning)
        out_channels       = 2,
        embedding_type     = 'positional',
        encoder_type       = 'residual',
        decoder_type       = 'standard',
        channel_mult_noise = 2,
        resample_filter    = [1, 3, 3, 1],
        channel_mult       = [2, 2, 2],
        attn_resolutions   = [32],
        time_emb           = 1,                 # must be 1 for lead-time conditioning
    )


In [2]:
model.enc

ModuleDict(
  (64x64_conv): Conv2d()
  (64x64_block0): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
    (skip): Conv2d()
  )
  (64x64_block1): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
  )
  (64x64_block2): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
  )
  (64x64_block3): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
  )
  (32x32_down): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
    (skip): Conv2d()
  )
  (32x32_aux_residual): Conv2d()
  (32x32_block0): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Con

In [3]:
model.dec

ModuleDict(
  (16x16_in0): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
    (norm2): GroupNorm()
    (qkv): Conv2d()
    (proj): Conv2d()
  )
  (16x16_in1): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
  )
  (16x16_block0): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
    (skip): Conv2d()
  )
  (16x16_block1): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
    (skip): Conv2d()
  )
  (16x16_block2): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): GroupNorm()
    (conv1): Conv2d()
    (skip): Conv2d()
  )
  (16x16_block3): UNetBlock(
    (norm0): GroupNorm()
    (conv0): Conv2d()
    (affine): Linear()
    (norm1): Gro

In [4]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable: {trainable}, Non-trainable: {non_trainable}")

Trainable: 56396674, Non-trainable: 0


In [8]:
from collections import defaultdict

sector_params = defaultdict(int)
sector_layers = defaultdict(set)
updown_params = defaultdict(int)
updown_layers = defaultdict(set)

for name, param in model.named_parameters():

    parts = name.split(".")
    if parts[0] in ("enc", "dec"):
        sector, layer = parts[0], parts[1]
    else:
        sector, layer = "mapping", parts[0]

    sector_params[sector] += param.numel()
    sector_layers[sector].add(layer)

    if sector in ("enc", "dec"):
        if "_down" in layer:
            key = (sector, "down")
        elif "_up" in layer:
            key = (sector, "up")
        else:
            key = None
        if key is not None:
            updown_params[key] += param.numel()
            updown_layers[key].add(layer)

print("-- sector summary ---")
for sector in sector_params:
    print(f"{sector}: layers={len(sector_layers[sector])}, params={sector_params[sector]}")

print("-- up/down summary ---")
for (sector, kind), params in updown_params.items():
    print(f"{sector} {kind}: layers={len(updown_layers[(sector, kind)])}, params={params}")

-- sector summary ---
mapping: layers=2, params=394240
enc: layers=17, params=19903616
dec: layers=21, params=36098818
-- up/down summary ---
enc down: layers=2, params=2756608
dec up: layers=2, params=2756608


In [2]:
dataset = SQGLeadTimeDataset(DATA_500)
dataloader = DataLoader(dataset, batch_size=64)
len(dataloader) * 64

760000

In [3]:
val_dataset = SQGLeadTimeDataset(DATA_500, split='val')
val_dataloader = DataLoader(val_dataset, batch_size=64)
len(val_dataloader) * 64

190016